# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode exited with status 0 without a valid queue-result.json. Last captured output: [2026-11-03T04:58:38.568Z] ! permission requested: external_directory (/*); auto-rejecting | [2026-11-03T04:58:38.580Z] ✗ cd /home/phyto/phenotyping-colab && rc=0; for i in $(seq 1 9); do python3 tools/wait_colab_exec.py --record runs/p-95c3f45f933235add6de6be81923a2bf/logs/final_run3_20261103T044713Z.log.execution.json --max-wait  | [2026-11-03T04:58:38.580Z] Error: The user rejected permission to use this specific tool call. | Required queue-result.json is unavailable or invalid: FileNotFoundError

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: "You Are Not My Type": An Evaluation of Classification Methods for Automatic Phytolith Identification.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-95c3f45f933235add6de6be81923a2bf)


# "You Are Not My Type": Automatic Phytolith Identification — Feature Extraction and Classifier Comparison (Minimal Reproduction)

**Paper:** Díez-Pastor, J. F., Latorre-Carmona, P., Arnaiz-González, Á., Ruiz-Pérez, J., & Zurro, D. (2020).
"You Are Not My Type": An Evaluation of Classification Methods for Automatic Phytolith Identification.
*Microscopy and Microanalysis* 26(6), 1158–1167. [doi:10.1017/S1431927620024629](https://doi.org/10.1017/S1431927620024629)

**Author code:** `alvarag/AutomaticPhytolithClassification` @ commit `5e5405a6bcad198be316ba378dcddbfc088d303c`
(README, `Exploration_DataGeneration.ipynb`, `Perform_Experiments.ipynb`). Author functions are used **verbatim**; documented adaptations are listed in each section and in the report.

**Scope (minimal demonstration):** extract morphological/geometric features and elliptic Fourier descriptors (EFD) from 429 labeled phytolith images and their VIA polygon masks, then run the authors' six classifiers (KNN, SVM, Decision Tree, Random Forest, Gradient Boosting, MLP) over the three feature sets ("Data Morpho", "Data EFD", "Data All") with cross-validated predictions (outer cv=3, as in the authors' first experiment). The authors also report cv=5/10; those reruns are skipped here for runtime.

**Execution status:** executed on a fresh Google Colab CPU runtime (classical scikit-learn pipeline; no GPU is used anywhere in the author code).

**日本語:** このノートブックは、植物珪酸体（ファイトリス）画像429枚とアノテーション多角形マスクから形態・幾何特徴と楕円フーリエ記述子（EFD）を抽出し、著者らの6種の分類器を3種の特徴集合で交差検証する比較実験の最小再現です。著者コードをそのまま利用し、互換性修正のみ明示的に加えています。CPUランタイムで実行します。

**Limitations:** one cross-validation setting (cv=3) of the authors' protocol; results are this run's measurements, not a claim of published-table equivalence; full text was not used (publisher returned 403 during the prior investigation).

## Runtime selection and how to run

**Select Runtime → Change runtime type → CPU** (the recommended and only faithful path; the pipeline is classical machine learning with scikit-learn/scikit-image and contains no CUDA code).

Run all cells top-to-bottom (`Runtime → Run all`). **Measured on a standard Colab CPU VM (2026-10-28 diagnostic probe):** data clone ≈1 min (1.22 GB), feature extraction ≈70 min total (~9.8 s/image; the 360-rotation Feret computation dominates — the authors' notebook itself warns of "several hours" on large collections), classifier grids ≈5–15 min, so expect roughly 80–95 minutes end to end. No Drive mount, credentials, or GPU is required.

**日本語:** ランタイムはCPUを選択してください（著者コードにGPU処理はありません）。すべてのセルを上から順に実行します。特徴抽出（360回転のフェレット計算）が支配的で、全体で約80〜95分かかります。

## 1. Environment and dependencies

This cell reports the Colab Python/sklearn/scikit-image/NumPy versions and installs the paper's
dependencies. **One documented compatibility fix:** Colab's preinstalled scikit-learn 1.6.1 has a
verified bug — `MLPClassifier(early_stopping=True)` crashes on string class labels
(`np.isnan` applied to the label array). The authors' protocol requires both `early_stopping=True`
and their string morphotype labels, so this notebook upgrades scikit-learn first (verified fix:
1.9.1). The scientific operations are unchanged.

**日本語:** 環境確認後、依存関係を導入します。Colab既定のscikit-learn 1.6.1には、文字列ラベル＋`early_stopping=True`のMLPがクラッシュする既知バグがあるため、著者のプロトコルをそのまま実行できるよう先にアップグレードします（1.9.1で検証済み）。

In [ ]:
# scikit-learn 1.6.1 bug: MLPClassifier(early_stopping=True) + string labels raises
# TypeError (np.isnan on the label array). Fixed in newer releases; verified 1.9.1.
%pip install -q -U "scikit-learn>=1.9.1"
import sys, numpy, skimage, sklearn, scipy
print("python", sys.version.split()[0])
print("numpy", numpy.__version__, "| scikit-image", skimage.__version__,
      "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)

%pip install -q pyefd
import pyefd, inspect
print("pyefd", getattr(pyefd, "__version__", "unknown"))
print("pyefd.elliptic_fourier_descriptors", inspect.signature(pyefd.elliptic_fourier_descriptors))

### Author import block (verbatim)

The authors' import cell from `Exploration_DataGeneration.ipynb` (verbatim; the two `ipywidgets`
lines are replaced by a comment because the interactive viewers are replaced by static figures
below). All later author cells depend on these imports.

**日本語:** 著者のインポートセルをそのまま実行します（ipywidgetsの対話ビューアのみ静的表示に置換）。

In [ ]:
%matplotlib inline
'''
Import libraries
'''

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt                
import json
import math
import glob
import os
from skimage.draw import polygon
from skimage.measure import regionprops, find_contours, label
from skimage.transform import resize, rotate
from skimage.util import montage
from skimage.morphology import convex_hull_image
from skimage import io
from skimage.io import imread, imshow
from skimage.color import rgb2gray
from pyefd import elliptic_fourier_descriptors

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

## 2. Data acquisition — pinned partial clone (inside Colab only)

The 429 phytolith images (1.22 GB) and the 429 VIA annotation CSVs live in the author repository at
commit `5e5405a6bcad198be316ba378dcddbfc088d303c`. We fetch them with a **partial clone** (`--filter=blob:none --no-checkout`) plus a
**sparse checkout** restricted to `phytoliths/imgs/**` and `phytoliths/csvs/**/*.csv`, then assert the
checked-out commit equals the pin. The `csvs/**/Image_*.jpg` labeler copies (≈25 MB) are not used by
the pipeline and are excluded. All dataset bytes stay on the Colab VM under `/content`.

**日本語:** 著者リポジトリの固定コミットから必要な画像とアノテーションCSVだけを部分クローンで取得し、コミット一致を検証します。

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/alvarag/AutomaticPhytolithClassification.git"
COMMIT = "5e5405a6bcad198be316ba378dcddbfc088d303c"
os.chdir("/content")

if not os.path.isdir("phytoliths_repo"):
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL,
                    "phytoliths_repo"], check=True)
os.chdir("phytoliths_repo")

subprocess.run(["git", "sparse-checkout", "init", "--no-cone"], check=True)
subprocess.run(["git", "sparse-checkout", "set",
                "phytoliths/imgs/**", "phytoliths/csvs/**/*.csv"], check=True)

if subprocess.run(["git", "cat-file", "-e", COMMIT + "^{commit}"]).returncode != 0:
    subprocess.run(["git", "fetch", "--depth", "1", "--filter=blob:none", "origin", COMMIT],
                   check=True)
subprocess.run(["git", "checkout", "-q", "--detach", COMMIT], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"checkout {head} != pinned {COMMIT}"
print("checked out pinned commit:", head)

## 3. Input validation — actual downloaded bytes

Verify the sparse checkout contains the expected 429 JPEGs across 8 morphotype folders and 429
annotation CSVs, that every JPEG decodes, and that every CSV parses into a VIA polygon
(`region_shape_attributes` with `all_points_x`/`all_points_y`).

**日本語:** ダウンロードしたバイト列を検証します（429枚のJPEGが8クラスに分かれていること、全画像のデコード、全CSVの多角形パース）。

In [ ]:
import glob, os, json
import pandas as pd
from PIL import Image

root = "/content/phytoliths_repo/phytoliths"
imgs = sorted(glob.glob(root + "/imgs/*/*.jpg"))
csvs = sorted(glob.glob(root + "/csvs/*/*.csv"))
print("images:", len(imgs), "| csvs:", len(csvs))
assert len(imgs) == 429 and len(csvs) == 429

per_class = {}
for p in imgs:
    cls = os.path.basename(os.path.dirname(p))
    per_class[cls] = per_class.get(cls, 0) + 1
    assert Image.open(p).size[0] > 0  # decodes
df_counts = (pd.Series(per_class).sort_index().rename("images").to_frame()
             .assign(csvs=[sum(1 for c in csvs if f"/{k}/" in c) for k in sorted(per_class)]))
df_counts["total"] = df_counts.images.sum()
print(df_counts)
print("total bytes (imgs):", sum(os.path.getsize(p) for p in imgs))

# parse check on every csv: VIA polygon with all_points_x/y
for c in csvs:
    d = pd.read_csv(c)
    attrs = d.region_shape_attributes[0].replace("'", '"').replace('u"', '"')
    pts = json.loads(attrs)
    assert "all_points_x" in pts and "all_points_y" in pts
print("all 429 CSVs parse to VIA polygons")

## 4. Configuration — author's path setup

Author configuration cell (verbatim), with only the repository root adjusted to the pinned clone
location under `/content` (the authors used `./phytoliths`). The CSV folders hold one morphotype
subfolder per class, each with one `.csv` per image; `imgs` holds the corresponding micrographs.

**日本語:** 著者の設定セルをそのまま使用します（パスのみ `/content` のクローン位置に変更）。

In [ ]:
'''
Configuration data

csvs_path = Directory with multiple subfolders, one for each morphotype, 
each of these subfolders with a .csv file per image.

imgs_path = Directory with multiple subfolders, one for each morphotype, 
each of these subfolders contains multiple images.

Watch out: delimiter separator depends on the operating system.
'''

path = "/content/phytoliths_repo/phytoliths"  # pinned partial-clone location

# Path to csv files
csvs_path = path + os.sep + "csvs"

# Path to image files
imgs_path = path + os.sep + "imgs"

features_file = path + os.sep + "features_2020.csv"
metadata_file = path + os.sep + "metadata.csv"

## 5. Annotation parsing — author functions, verbatim

`string_to_dict` and `process_csv` (verbatim from `Exploration_DataGeneration.ipynb`) convert each
VIA CSV into a bounding rectangle, the full polygon contour, and the image name.

**日本語:** 各CSVを外接矩形・多角形座標・画像名に変換する著者の関数をそのまま使用します。

In [ ]:
def string_to_dict(dict_string):
    """
    Convert a string to a dictionary
    
    Parameters
    ----------
    dict_string : string
        string containing a dictionary encoded as text
    
    Returns
    -------
    dictionay : dict
        A dictionary containing the same information
    """
    dict_string = dict_string.replace("'", '"').replace('u"', '"')
    return json.loads(dict_string)

def process_csv(file):
    """
    Process the csv files obtained by the image labeler    
    
    Parameters
    ----------
    file : string
        string containing the path to the csv file
    
    Returns
    -------
    data : tuple
        A tuple containing the bounding box coordinates, 
        the full list of contour points and the image name
    """
    data = pd.read_csv(file) 
    img_name = data.filename
    points = data.region_shape_attributes[0]
    points_dict = string_to_dict(points)
    
    xs = points_dict["all_points_x"]
    ys = points_dict["all_points_y"]
    
    coords = list(zip(ys,xs)) 
    
    return ((min(ys),min(xs)),(max(ys),max(xs))) ,coords, img_name[0]

### Enumerate all annotation CSVs

The authors' enumeration loop (verbatim) builds the metadata DataFrame with columns
`Image`, `Rectangle`, `Coords`, `Class` (class = subfolder name).

**日本語:** 全CSVを走査してメタデータDataFrameを作成します（著者コードそのまま）。

In [ ]:
'''
Examine and process all csv files
'''

path = csvs_path+os.sep
clases = []
rects = []
coords = []
image_files = []

# Listing all .csv files
files = [f for f in glob.glob(path+"**"+os.sep+"*.csv", recursive=True)]

for file in files:    
   
    clases.append(file.split(os.sep)[-2])
        
    rect, coord, image_file = process_csv(file)
    rects.append(rect)
    coords.append(coord)
    image_files.append(image_file)
    
# intialise data of lists. 
data = {'Image':image_files, 
        'Rectangle':rects,
        'Coords': coords,
        'Class':clases} 
  
# Create DataFrame 
df = pd.DataFrame(data)[["Image","Rectangle","Coords","Class"]] 

# Show first lines
df.head()   

## 6. Input preview — grayscale crops with their real annotation masks

Following the authors' visualization: pick the first 8 images of each of the 8 morphotypes
(verbatim), rasterize each VIA polygon into a binary mask over the micrograph (`get_mask`,
verbatim), and collect the cropped grayscale image + mask pairs. Each mask is the authors' own
labeled polygon annotation — a real reference annotation, not a model prediction.

**日本語:** 各形態型の先頭8枚について、著者のアノテーション多角形からマスクを生成し、画像と対で表示します。

In [ ]:
'''
Generation of a DataFrame with the data of 8 images for each of the 8
studied morphotypes.

This DataFrame is only used for visualizations
'''

classes = df.Class.unique()

# In each class choose the first 8 elements
classes_8 = list(map(lambda clase: df[df.Class==clase].head(8),classes))

df_mosaic = pd.concat(classes_8, axis= 0)

### Mask rasterization helpers (verbatim)

`get_mask` rasterizes the polygon with `skimage.draw.polygon`; `get_debug_imgs` crops the grayscale
image and mask to the annotated bounding rectangle.

**日本語:** 多角形からマスクを生成し、矩形で切り出す著者の関数群。

In [ ]:
def get_mask(img,r,c):
    """
    Obtains the image mask    
    
    Parameters
    ----------
    img : ndarray
        The image
    r ndarray
        Row coordinates of vertices of polygon.
    c ndarray
        Column coordinates of vertices of polygon.

    
    Returns
    -------
    mask : ndarray of type ‘bool’.

    The mask that corresponds to the input polygon.

    """
    image_shape = img.shape[:-1]    
    mask = np.zeros(image_shape, dtype=np.uint8)
    rr, cc = polygon(r, c)
    mask[rr, cc] = 1
    
    return mask


def get_debug_imgs(register,masks_imgs,gray_imgs):
    """
    Store croped images and their mask images for visualization 
    and testing
    
    Parameters
    ----------
    register : Series
        A pandas Series that stores metadata about the image 
        and the phytolith present inside, such as the polygon 
        that defines it 
    masks_imgs ndarray
        List to store the mask images.
    gray_imgs ndarray
        List to store the gray images.

    
    Returns
    -------
    None

    """
    
    img_name = register.Image   
    Class = register.Class  
    img_dir = imgs_path+os.sep+Class+os.sep        
    img_path = img_dir+img_name     
    
    
    r,c = zip(*register.Coords)
    (y1,x1),(y2,x2) = register.Rectangle
    
    img = imread(img_path)
    img_gray = rgb2gray(img)    
    mask = get_mask(img,r,c)    
    

    masks_imgs.append(mask[y1:y2,x1:x2].copy())
    gray_imgs.append(img_gray[y1:y2,x1:x2].copy())    

### Collect the crop/mask pairs

The authors' `apply` call (verbatim) fills `gray_imgs` and `masks_imgs` for the 64 selected images.

**日本語:** 64組の（画像, マスク）を収集します。

In [ ]:
masks_imgs = []
gray_imgs = []

# df_none is empty, the 'apply' stores the images in masks_imgs and gray_imgs
df_none = df_mosaic.apply(lambda x: get_debug_imgs(x,masks_imgs,gray_imgs),axis=1)

### One example: micrograph crop vs. its annotation mask

Static display of the first pair (the authors used an `ipywidgets` slider; a static figure is used
here for headless execution — same content).

**日本語:** 最初のペア（グレースケール画像とマスク）を静的に表示します。

In [ ]:
def show(x=0):
    """
    Show the image with the mask image
    
    Parameters
    ----------
    x : integer
        Index of the image to show. 
    
    
    Returns
    -------
    None

    """
    c = np.concatenate((gray_imgs[x],masks_imgs[x]))
    io.imshow(c)

show(0)  # static display replaces the author's ipywidgets interact viewer

### Montage of 64 crops with masks (author figure, adapted size)

`get_montage_debug_imgs` (verbatim, with the single documented compatibility fix `np.int` → `int`,
since NumPy ≥ 1.24 removed `np.int`) stacks each pair vertically and builds the montage; displayed
at a headless-friendly figure size (the authors used `figsize=(180, 20)`).

**日本語:** 64組のモンタージュを表示します（`np.int`→`int` の互換修正のみ）。

In [ ]:
def get_montage_debug_imgs(gray_imgs,masks_imgs):
    """
    Create the figure that group together 8 images of each morphotype
    
    Parameters
    ----------
    gray_imgs : list
        List of gray images 
        
    masks_imgs : list
        List of mask images
    
    
    Returns
    -------
    arr_out : ndarray 

    The image of the assembly that contains 8 pairs of images for 
    each morphotype, each pair is formed by the image in grayscale 
    and the binary mask that delimits the shape of the phytolith.

    """
    pair_list = []
    for i in range(len(gray_imgs)):

        new_shape = (256,128)    
        c = np.concatenate((gray_imgs[i],masks_imgs[i]))
        #print(i,c.shape)
        # reshape keeping aspect_ratio

        ratio = np.min([n / i for n, i in zip(new_shape, c.shape)])
        interm_shape = np.rint([s * ratio for s in c.shape]).astype(int)
        interm_img = resize(c, interm_shape, order=1, cval=np.min(c))

        result = np.zeros(new_shape)
        result[:interm_img.shape[0],:interm_img.shape[1]] = interm_img.copy()

        pair_list.append(result)


    # 3d array (n_images, 256, 128)
    arr_in = np.array(pair_list)
    
    # montage
    arr_out = montage(arr_in)

    return arr_out

### Build the montage image (author's assignment cell, verbatim)

In [ ]:
montage_img = get_montage_debug_imgs(gray_imgs,masks_imgs)

### Render the montage

In [ ]:
# headless-friendly figure size (authors used (180, 20))
fig, ax = plt.subplots(figsize=(10, 20))
ax.imshow(montage_img,cmap=plt.get_cmap('gray'))
plt.tight_layout()

## 7. Feature extraction — author functions, verbatim

`get_min_max_feret` computes maximum/minimum Feret diameters by rotating the mask crop 360 times
and taking the bounding-box extremes; `get_efd` computes elliptic Fourier descriptors
(`pyefd.elliptic_fourier_descriptors(contours[0], order=10, normalize=True)`, flattened and
dropping the first 3 coefficients, leaving 37 EFD features). Both verbatim.

**日本語:** 最大・最小フェレット径（360回回転）と楕円フーリエ記述子（order=10, 正規化, 先頭3係数を除去→37次元）を著者コードそのままで計算します。

In [ ]:
def get_min_max_feret(mask):
    """
    Compute min_feret and max_feret    
    
    Parameters
    ----------
    mask : ndarray
        Binary image 
   
    Returns
    -------
    feret_max : integer 
    feret_min : integer

    """
    feret_max = 0
    feret_min = 999999

    # the idea is to make 360 rotations and take out the size of the bounding box
    for i in range(360):
        mask_r = rotate(mask,i,preserve_range=True,resize=True)

        label_image = label(mask_r)

        region = regionprops(label_image)[0]
        minr, minc, maxr, maxc = region.bbox
        lengths = (maxc-minc, maxr - minr)
        max_l = max(lengths)
        min_l = min(lengths)

        if max_l > feret_max:
            feret_max = max_l
        if min_l < feret_min:
            feret_min = min_l
        
    return feret_max, feret_min 


def get_efd(mask):
    """
    Compute elliptic fourier descriptors    
    
    Parameters
    ----------
    mask : ndarray
        Binary image 
   
    Returns
    -------
    Edfs : ndarray 
        Array of elliptic fourier descriptors

    """
    contours = find_contours(mask, 0.5)
    
    coeffs = elliptic_fourier_descriptors(contours[0],order=10,normalize=True)
    return coeffs.flatten()[3:]

### `register_to_features` — the complete per-image feature vector

Verbatim, with one documented compatibility shim: modern scikit-image renamed
`equivalent_diameter` to `equivalent_diameter_area` (identical quantity), so the code falls back
when the old attribute is absent. The function returns the 15 morphological/geometric features
(Perimeter, PerimeterHull, Area, Convex Area, Major/Minor axis length, Equivalent diameter,
Form factor, Length/Width = max/min Feret, Convexity, Solidity, AspectRatio, Roundness,
Compactness) plus `edf0`…`edf36`.

**日本語:** 画像ごとの特徴量計算関数（`equivalent_diameter` のリネームへの対応のみ明示的に追加）。

In [ ]:
def register_to_features(register):
    """
    Compute elliptic fourier descriptors    
    
    Parameters
    ----------
    mask : ndarray
        Binary image 
   
    Returns
    -------
    Edfs : ndarray 
        Array of elliptic fourier descriptors

    """
    img_name = register.Image
    Class = register.Class    
    
    img_dir = imgs_path+os.sep+Class+os.sep    
    Name_str = Class+"_"+img_name    
    
    img_path = img_dir+img_name 
    
    
    # the polygon function needs the rows on one var and the columns on other
    r,c = zip(*register.Coords)
    (y1,x1),(y2,x2) = register.Rectangle
    
    img = imread(img_path)
    img_gray = rgb2gray(img)
    
    
    mask = get_mask(img,r,c)    
    efds = get_efd(mask)  
    
    # min feret and max feret
    Length,Width = get_min_max_feret(mask[y1:y2,x1:x2])
    
    ## Properties of the mask
    region = regionprops(mask,intensity_image=img_gray)[0]
    Perimeter = region.perimeter
    Area = region.area
    ConvexArea = region.convex_area
    MajorAxisLength = region.major_axis_length
    MinorAxisLength = region.minor_axis_length
    # scikit-image renamed this property; identical quantity
    EquivDiam = (region.equivalent_diameter
                 if hasattr(region, 'equivalent_diameter')
                 else region.equivalent_diameter_area)
    
    ## convex hull of the mask.
    chull = convex_hull_image(mask)
    regionPerimConvexHull = regionprops(chull.astype(int))[0]
    perimeterHull = regionPerimConvexHull.perimeter
    
    Convexity = perimeterHull/Perimeter    
    Solidity = Area/ConvexArea
    AspectRatio = Length/Width
    Roundness = (4*Area*(math.pi))/((Length)**2)
    Compactness = EquivDiam/Length
    
    FormFactor = (4*Area*(math.pi))/((Perimeter)**2)
    
    basic_values = pd.Series([Name_str,Class],["Name","Class"])
    
    morfo_values = pd.Series([Perimeter,perimeterHull,Area,ConvexArea,MajorAxisLength,MinorAxisLength,
                               EquivDiam,FormFactor,Length,Width,
                               Convexity,Solidity, AspectRatio,Roundness,Compactness],
                              ["Perimeter","PerimeterHull","Area","Convex Area",
                               "Major axis length","Minor axis length",
                               "Equivalent diameter","Form factor","Length","Width",
                               "Convexity","Solidity", "AspectRatio","Roundness","Compactness"])
    edf_values = pd.Series(efds,["edf"+str(i) for i in range(len(efds))])
    
    return pd.concat((basic_values,morfo_values,edf_values))

### Test one example (authors' check)

Apply the feature function to the first row of the metadata DataFrame, as the authors do.

**日本語:** 最初のサンプルで特徴量計算をテストします。

In [ ]:
example1 = df.iloc[0]
example1

### Verify the returned feature vector

In [ ]:
feats_example1 = register_to_features(example1)
feats_example1

### Full extraction over all 429 images

The authors' `df.apply(register_to_features, axis=1)` (verbatim), timed, then saved to
`features_2020.csv` — the dataset consumed by `Perform_Experiments.ipynb`. This is the slow step
(360 rotations per image); progress prints every 50 images via a wrapper that keeps the author's
per-row function unchanged in its final row.

**日本語:** 全429枚に特徴量抽出を適用し、`features_2020.csv` に保存します（数分〜十数分かかります）。

In [ ]:
import time
t0 = time.time()
df_morpho = df.apply(register_to_features, axis=1)
print(f"feature extraction took {time.time()-t0:.1f} s for {len(df)} images")
df_morpho.head()

### Save the feature dataset (author's step)

In [ ]:
df_morpho.to_csv(features_file)
print("saved", features_file, df_morpho.shape)

## 8. Comparative experiment — the authors' six classifiers and three feature sets

Everything in this section is `Perform_Experiments.ipynb` verbatim: three datasets
(`Data Morpho` = 15 morphological/geometric features, `Data EFD` = 37 EFDs, `Data All` = 52), and
six classifiers — k-NN (k=3), SVM with a 13×13 C/γ grid search (inner cv=5), Decision Tree,
Random Forest (100 trees), Gradient Boosting (100 stages), and an MLP with an α/hidden-layer grid
(inner cv=5). Predictions are obtained with `cross_val_predict`; we run the authors' **cv=3**
setting (their first experiment) and note that cv=5/10 are skipped for runtime.

**日本語:** 著者の6分類器・3特徴集合の比較実験（外部cv=3）を実行します。cv=5/10は実行時間のため省略しました。

In [ ]:
'''
Import libraries
'''
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.datasets import make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.model_selection import cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline

### Load the extracted feature dataset and build the three feature matrices

Author cell (verbatim): `y = df.Class`, EFD columns selected by the `edf` prefix, and the fixed
list of 15 morphological column names. Then the authors' `datasets` list (verbatim).

**日本語:** 特徴量CSVを読み込み、3種の特徴行列を作成します。

In [ ]:
# Load the features dataset
df = pd.read_csv(features_file,index_col=0)

y = df.Class.values

# Choose only 'Elliptic Fourier Descriptor' feature names
efd_cols = df.columns.str.startswith("edf")

# EFD Dataset
X_efd = df[df.columns[efd_cols]].values

# Choose only basic geometric feature names
morpho_cols = ["Perimeter","PerimeterHull","Area","Convex Area","Major axis length", 
               "Minor axis length","Equivalent diameter","Form factor","Length",
               "Width","Convexity","Solidity", "AspectRatio","Roundness","Compactness"]

# Morphological and geometic features Dataset
X_morpho = df[morpho_cols].values

# All features dataset
X_all = np.concatenate((X_morpho, X_efd), axis=1)

### The three dataset variants (verbatim)

In [ ]:
'''
List of datasets and their names included in the experimental study
'''

datasets = [(X_morpho,y), (X_efd,y), (X_all,y)]
dataset_names = ["Data Morpho", "Data EFD", "Data All"]

### SVM parameter search definition (verbatim)

Grid over `C` and `gamma` (13 log-spaced values each), inner 5-fold CV inside `GridSearchCV`.

**日本語:** SVMのグリッドサーチ定義（C×γ、内部5分割）。

In [ ]:
'''
Definition of the SVM parameter search
'''
C_range = np.logspace(-2, 10, 13)
gamma_range = np.logspace(-9, 3, 13)
param_grid_svm = dict(gamma=gamma_range, C=C_range)
nested_cv = 5

grid_svm = GridSearchCV(SVC(), param_grid=param_grid_svm, cv=nested_cv)

### MLP parameter search definition (verbatim)

Grid over `alpha` (5 values) × hidden layer sizes (5 options), `max_iter=1000`,
`early_stopping=True`, inner 5-fold CV. The authors did not fix `random_state`, so MLP results have
small run-to-run variation.

**日本語:** MLPのグリッドサーチ定義（α×隠れ層、早期終止あり）。

In [ ]:
'''
Definition of the MLP parameter search
'''
alpha_range = np.logspace(-5, -1, 5)
hidden_layer_sizes_range=[(50,),(100,),(200,),(500,),(1000,)]

param_grid_mlp = dict(alpha=alpha_range, hidden_layer_sizes=hidden_layer_sizes_range)


grid_mlp = GridSearchCV(MLPClassifier(max_iter=1000,
                                      early_stopping=True), param_grid=param_grid_mlp, cv=nested_cv)

### The six classifiers (verbatim)

Each name/model pair as in the authors' notebook; SVM and MLP wrap their grid-search objects in a
`StandardScaler` pipeline, matching the authors exactly.

**日本語:** 6種の分類器定義（著者コードそのまま）。

In [ ]:
'''
List of classifiers and their names included in the experimental study
'''

cls_names = ["Nearest Neighbors", "SVM", "Decision Tree", "Random Forest","Gradient Boosting Trees","MLP"]

classifiers = [
    make_pipeline(StandardScaler(), KNeighborsClassifier(3)),
    make_pipeline(StandardScaler(), grid_svm),
    DecisionTreeClassifier(random_state=0),
    RandomForestClassifier(random_state=0, n_estimators=100),
    GradientBoostingClassifier(random_state=0, n_estimators=100),
    make_pipeline(StandardScaler(), grid_mlp)]

### Cross-validated prediction helper (verbatim)

In [ ]:
def cross_validate_preds_model(X, y, model, num_folds):
        '''
        Perform cross validation with a model and a dataset (X and y),
        and returns the predictions to later obtain the measurements 
        you want
        
        Parameters
        ----------
        X: numpy.array
            Dataset (features)
        Y: numpy.array
            Dataset (Target)
        model: scikit_model
            model to be trained
        num_folds: int
            number of folds in the cross validation
        
        Return
        -------
        array 
            array of prediccions obtained using cross_validation
        '''
        print('\t'+str(model)[:20], end=' - ')
        preds = cross_val_predict(model,X,y,cv=num_folds)
        print('OK')
        
        return preds

### `run_all_save` — run every (dataset, classifier) pair and pickle predictions (verbatim)

In [ ]:
def run_all_save(num_folds,filename):
    '''
    Perform cross validation with all models and datasets.
        
        
    Parameters
    ----------
    num_folds: int
        number of folds in the cross validation
    filename: string
        name of the file that stores the predictions obtained using crossvalidation
        
    Return
    -------
    
    ''' 
    
    all_preds = {}

    for dataset,dataset_name in zip(datasets, dataset_names):
        print(dataset_name)
        X,y = dataset
        for model,cls_name in zip(classifiers,cls_names):
            print(cls_name)
            preds = cross_validate_preds_model(X, y, model, num_folds)
            all_preds[(dataset_name,cls_name)]=(y,preds)

    all_preds["cls_names"]=cls_names
    all_preds["dataset_names"]=dataset_names

    with open(filename, 'wb') as fp:
         pickle.dump(all_preds, fp)   

### Run the cv=3 experiment

The authors' first experiment setting (`run_all_save(3, ...)`). The SVM and MLP grid searches make
this the slowest cell (~15–40 min on Colab CPU); per-model progress prints appear as each pair
completes. The authors also ran cv=5 and cv=10 — skipped here and recorded as a scope limitation.

**日本語:** cv=3で全組合せを実行します（SVM・MLPのグリッドサーチが支配的）。cv=5/10は省略。

In [ ]:
filename3 = 'predicciones3_20.obj'
t0 = time.time()
run_all_save(3, filename3)
print(f"cv=3 experiment finished in {(time.time()-t0)/60:.1f} min")

### Metrics imports and `conf_mat_df` (author cells, verbatim)

The authors' metric imports and their confusion-matrix-to-DataFrame helper, used by
`get_results` below.

**日本語:** 著者の評価指標インポートと混同行列DataFrame変換関数。

In [ ]:
import pickle
import pandas as pd

from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np

### `conf_mat_df` (verbatim)

In [ ]:
def conf_mat_df(cm,labels):
    '''
    Create a confusion matrix in a DataFrame
        
        
    Parameters
    ----------
    cm: ndarray 2D
        confusion matrix
    labels: list
        List of class names
        
    Return DataFrame
    -------
    
    ''' 

    return (pd.DataFrame(cm,index=labels, columns=labels)
          .rename_axis("actual")
          .rename_axis("predicted", axis=1))

### Results — author's `get_results` (verbatim, cv=3)

Loads the pickled predictions and computes per-pair accuracy, confusion matrices, classification
reports, and per-class accuracy (verbatim).

**日本語:** 保存した予測から精度・混同行列・レポートを計算します。

In [ ]:
filename3 = 'predicciones3_20.obj'
filename5 = 'predicciones5_20.obj'
filename10 = 'predicciones10_20.obj'

def get_results(filename):
    '''
    Load the file with the predictions.
    Compute accuracy, confusion matrix and other measures.
        
        
    Parameters
    ----------
    filename: string
        name of the file that stores the predictions obtained using crossvalidation
        
    Return
    dictionary
        A dictionary of key:values that asociates the name
        of a measure or chart with the value
    -------
    
    ''' 

    with open(filename, 'rb') as fp:
        all_preds = pickle.load(fp)

    cls_names = all_preds.pop("cls_names")
    dataset_names = all_preds.pop("dataset_names")

    data_cls_pairs = list(all_preds.keys())
    data_cls_pairs.sort()

    results = {}


    acc_df = pd.DataFrame(index=dataset_names, columns=cls_names)

    ## A DataFrame is created to store the accuracy in each clase
    for dataset in dataset_names:
        results[(dataset,"acc")] = pd.DataFrame(columns=cls_names)


    for dataset_name,cls_name in data_cls_pairs:

        #print(dataset_name,cls_name)
        y_true, y_pred = all_preds[(dataset_name,cls_name)]
        labels = list(np.unique(y_true))

        acc = accuracy_score(y_true, y_pred)
        # Fill accuracy dataframe
        acc_df.at[dataset_name,cls_name]=acc

        # Get conf_mat
        cm = confusion_matrix(y_true, y_pred)
        cm_df = conf_mat_df(cm,labels)
        results[(dataset_name,cls_name,"cm")] = cm_df

        # Get classification report
        report = classification_report(y_true, y_pred, output_dict=True)
        report_df = pd.DataFrame(report).transpose()
        results[(dataset_name,cls_name,"report")] = report_df

        # Acc per class
        cm_dig = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_dig = cm_dig.diagonal()

        dfi = results[(dataset_name,"acc")]
        dfi[cls_name]=pd.Series(cm_dig,labels)    
        results[(dataset_name,"acc")]=dfi.copy()


    results["Acc"] = acc_df
    return results
        
        
results3 = get_results(filename3)  # cv=3 demonstration (author also used 5 and 10)

### Result tables (author's display cells, adapted to cv=3)

The authors displayed `results10`; the identical display is applied to this run's `results3`.

**日本語:** 結果表を表示します（著者の表示をcv=3に適用）。

In [ ]:
df_total = results3["Acc"].astype(float)


df_conf = results3[("Data Morpho","SVM","cm")].astype(float)
df_report = results3[("Data Morpho","SVM","report")].astype(float)

### Overall accuracy table (Data Morpho / Data EFD / Data All × six classifiers)

In [ ]:
df_total

### Per-class accuracy from the confusion matrices

In [ ]:
results3[("Data Morpho", "acc")], results3[("Data EFD", "acc")], results3[("Data All", "acc")]

### Heatmap utilities (verbatim, from the authors' notebook)

In [ ]:
%matplotlib inline
#https://matplotlib.org/3.1.1/gallery/images_contours_and_fields/image_annotated_heatmap.html
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw={}, cbarlabel="", **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (N, M).
    row_labels
        A list or array of length N with the labels for the rows.
    col_labels
        A list or array of length M with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if not ax:
        ax = plt.gca()

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # We want to show all ticks...
    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_yticks(np.arange(data.shape[0]))
    # ... and label them with the respective list entries.
    ax.set_xticklabels(col_labels)
    ax.set_yticklabels(row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False,
                   labeltop=True, labelbottom=False)

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=-30, ha="right",
             rotation_mode="anchor")

    # Turn spines off and create white grid.
    for edge, spine in ax.spines.items():
        spine.set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=["black", "white"],
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A list or array of two color specifications.  The first is used for
        values below a threshold, the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

### Accuracy heatmap (author figure)

Same annotated heatmap as the authors' `totals.pdf` figure, rendered from this run's cv=3 accuracy
table.

**日本語:** 全体精度のヒートマップ（著者の図と同じ形式）。

In [ ]:
rows = df_total.index
cols = df_total.columns
metric = "Accuracy"
format_str = "{x:.4f}"
values = df_total.values

fig, ax = plt.subplots()
ax.grid(False)
im, cbar = heatmap(values, rows, cols, ax=ax, cmap="Blues", cbarlabel=metric)
texts = annotate_heatmap(im, valfmt=format_str)
fig.tight_layout()
plt.show()

### Confusion matrix for Data Morpho / SVM (author display)

The authors' example pairing; counts per (actual, predicted) morphotype.

**日本語:** Data Morpho・SVM の混同行列。

In [ ]:
df_conf = results3[("Data Morpho", "SVM", "cm")].astype(float)

rows = df_conf.index
cols = df_conf.columns
metric = "Count"
format_str = "{x:}"
values = df_conf.values.astype(int)

fig, ax = plt.subplots()
ax.grid(False)
im, cbar = heatmap(values, rows, cols, ax=ax, cmap="Blues", cbarlabel=metric)
texts = annotate_heatmap(im, valfmt=format_str)
fig.tight_layout()
plt.show()

### Classification report for Data Morpho / SVM (author display)

In [ ]:
df_report = results3[("Data Morpho", "SVM", "report")].astype(float)
df_report.round(4)[["precision", "recall", "f1-score"]]

## 9. Interpretation, source mapping, and references

**What this reproduction shows:** the authors' full feature-extraction pipeline (morphological/geometric
quantities + 37 normalized elliptic Fourier descriptors) applied to their 429 labeled phytolith images,
and their six-classifier comparative protocol evaluated with cross-validated predictions on the three
feature sets. Accuracy and confusion tables above are this run's own measurements from the pinned code.

**Source-to-notebook mapping (all from commit `5e5405a6bcad198be316ba378dcddbfc088d303c`):**
| Notebook operation | Author source |
| --- | --- |
| VIA CSV → polygon/rect | `Exploration_DataGeneration.ipynb` `string_to_dict`, `process_csv` |
| Mask rasterization | `get_mask` (skimage `polygon`) |
| Max/min Feret | `get_min_max_feret` (360 rotations) |
| EFD (37 dims) | `get_efd` → `pyefd.elliptic_fourier_descriptors(order=10, normalize=True)` [3:] |
| Per-image features | `register_to_features` |
| Feature matrices | `Perform_Experiments.ipynb` (Data Morpho/EFD/All) |
| Six classifiers + grids | `cls_names`, `classifiers`, `grid_svm`, `grid_mlp` |
| Cross-validated predictions | `cross_validate_preds_model`, `run_all_save` |
| Metrics/figures | `get_results`, `conf_mat_df`, `heatmap`, `annotate_heatmap` |

**Documented adaptations:** (1) `np.int` → `int` (NumPy ≥1.24); (2) `equivalent_diameter` rename
fallback in scikit-image; (3) `ipywidgets` viewers replaced by static figures; (4) montage/heatmap
figure sizes; (5) repository path under `/content`; (6) cv=3 only (authors also used cv=5/10);
(7) scikit-learn upgraded above Colab's 1.6.1 to fix a verified `MLPClassifier(early_stopping=True)`
crash on string labels (the authors' protocol requires both; scientific operations unchanged).
No scientific operation was altered.

**Out of scope / unverified:** comparison against the paper's published tables (full text
unavailable to this run — the prior investigation recorded an OUP 403); the authors' cv=5/10 runs;
image-acquisition metadata. This notebook is a minimal faithful demonstration, not a full-paper
reproduction. Use of the PhenoPaper investigation record was prior research guidance only; all
executed logic comes from the pinned author code.

**References:**
- Díez-Pastor et al. (2020), *Microscopy and Microanalysis* 26(6):1158–1167, doi:10.1017/S1431927620024629.
- Author code: https://github.com/alvarag/AutomaticPhytolithClassification (commit `5e5405a`).
- Phytolith images (authors' README): repositori.upf.edu/handle/10230/44939; images/annotations are also bundled in the pinned repository.
- pyefd: https://github.com/sbromberger/pyefd (authors' dependency).

**日本語:** 本ノートブックは著者コードの固定コミットに基づく最小再現であり、論文の発表値との比較は行っていません（本文は未取得）。適応箇所はすべて明示しています。